In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

In [ ]:
!pip install -q "curatorkit[all]" aligntune "auditkit[transformers]"

In [ ]:
import torch
from huggingface_hub import login
from google.colab import userdata
login(userdata.get("HF_TOKEN"))

import curatorkit, aligntune, auditkit
print("kits ok | GPU:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForCausalLM
mid = "CohereLabs/tiny-aya-global"

# Added force_download=True to bypass and overwrite the corrupted cached files
tok = AutoTokenizer.from_pretrained(mid, force_download=True)
model = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=torch.float16, device_map="auto", force_download=True)

msgs = [{"role": "user", "content": "Reply with JSON: {\"status\": \"ok\"}"}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
print(tok.decode(model.generate(ids, max_new_tokens=30)[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# Define your prompt and query the loaded model
prompt = "Tell me about you."

# Apply the model's chat template and generate a response
inputs = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt").to(model.device)

# Properly extract the actual PyTorch tensor from the BatchEncoding dict-like object
outputs = model.generate(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, max_new_tokens=100, do_sample=True, temperature=0.7)

# Decode and print the assistant's reply
response = tok.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(response)

In [ ]:
print(model)
print(model.config)
total = sum(p.numel() for p in model.parameters())
print(f"parameters: {total/1e9:.2f} B")
for name, p in list(model.named_parameters())[:12]:
    print(name, tuple(p.shape))